# A/B-тест email-рассылки

**Бизнес-задача:** интернет-магазин проверил две рекламные рассылки - с мужскими и женскими товарами. Нужно определить, повысили ли они конверсию и выручку по сравнению с отсутствием письма.

**Данные:** 64 000 клиентов, рандомизированный эксперимент MineThatData (2008), опубликован Кевином Хиллстромом.

Источник: https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html

**Метрики:** основная - конверсия в покупку (`conversion`); дополнительные - посещение сайта (`visit`) и выручка на клиента (`spend`). Период наблюдения - 2 недели.

## 1. Загрузка данных
Устанавливаем библиотеку statsmodels и загружаем с общедоступного датасета данные

In [1]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from scipy.stats import chisquare, ttest_ind
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests

url = "https://raw.githubusercontent.com/AdityaDabrase/ab-testing-email-marketing/main/data/raw/hillstrom.csv"
df = pd.read_csv(url)
df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


## 2. Проверка данных и групп
Убеждаемся в размере датасета, количестве пропусков, а также количестве для каждого класса

In [3]:
print('Размер:', df.shape)
print('Пропуски:', df[['segment', 'conversion', 'visit', 'spend']].isna().sum())

print(df[df['segment']=='No E-Mail']['segment'].count())
print(df[df['segment']=='Mens E-Mail']['segment'].count())
print(df[df['segment']=='Womens E-Mail']['segment'].count())

Размер: (64000, 12)
Пропуски: segment       0
conversion    0
visit         0
spend         0
dtype: int64
21306
21307
21387


## 3. Метрики эксперимента
Сравниваем долю покупателей, долю посетителей и **среднюю выручку на одного назначенного клиента** (включая нулевые покупки).

In [4]:
df.groupby('segment').agg(
    users=('conversion', 'size'),
    conversion=('conversion', 'mean'),
    visit=('visit', 'mean'),
    revenue_per_user=('spend', 'mean')
).reindex(['No E-Mail', 'Mens E-Mail', 'Womens E-Mail'])


,users,conversion,visit,revenue_per_user
segment,,,,
No E-Mail,21306,0.005726,0.106167,0.652789
Mens E-Mail,21307,0.012531,0.182757,1.422617
Womens E-Mail,21387,0.008837,0.151400,1.077202


## 4. Статистические тесты

**H₀:** конверсия в группе рассылки равна конверсии без рассылки. Сравним каждую рассылку с контролем двусторонним z-тестом для долей и построим 95%-е ДИ разности конверсий. Поскольку сравнения два, применим поправку Холма.

Для выручки используем t-тест Уэлча по выручке **на всех клиентов**; эта метрика сильно скошена и содержит много нулей, поэтому её выводы считаем дополнительными, а не окончательным доказательством финансовой выгоды.

In [5]:
control = df[df['segment'] == 'No E-Mail']
rows = []

for name in ['Mens E-Mail', 'Womens E-Mail']:
    test = df[df['segment'] == name]
    successes = [test['conversion'].sum(), control['conversion'].sum()]
    sample_sizes = [len(test), len(control)]
    _, p_conversion = proportions_ztest(successes, sample_sizes)
    ci_low, ci_high = confint_proportions_2indep(
        successes[0], sample_sizes[0], successes[1], sample_sizes[1],
        compare='diff', method='newcomb'
    )
    _, p_revenue = ttest_ind(test['spend'], control['spend'], equal_var=False)
    rows.append({
        'variant': name,
        'conversion_lift_pp': 100 * (test['conversion'].mean() - control['conversion'].mean()),
        '95%_CI_pp': f'[{100 * ci_low:.2f}; {100 * ci_high:.2f}]',
        'p_conversion': p_conversion,
        'revenue_lift_per_user_$': test['spend'].mean() - control['spend'].mean(),
        'p_revenue_welch': p_revenue
    })

result = pd.DataFrame(rows)
result['p_conversion_holm'] = multipletests(result['p_conversion'], method='holm')[1]
result['p_revenue_holm'] = multipletests(result['p_revenue_welch'], method='holm')[1]
print(result.round(4).to_string(index=False))

      variant  conversion_lift_pp    95%_CI_pp  p_conversion  revenue_lift_per_user_$  p_revenue_welch  p_conversion_holm  p_revenue_holm
  Mens E-Mail              0.6805 [0.50; 0.86]        0.0000                   0.7698           0.0000             0.0000          0.0000
Womens E-Mail              0.3111 [0.15; 0.47]        0.0002                   0.4244           0.0011             0.0002          0.0011


## 5. Продуктовое решение

Положительный lift и `p_conversion_holm < 0.05` дают статистическое основание говорить о росте конверсии относительно контроля. Для решения о масштабировании дополнительно нужны стоимость рассылки, маржа, ограничения по частоте писем и оценка рисков. **Статистически значимый рост выручки не равен автоматически росту прибыли.**

In [6]:
for _, r in result.iterrows():
    if r['p_conversion_holm'] < 0.05 and r['conversion_lift_pp'] > 0:
        conclusion = 'рост конверсии статистически значим'
    else:
        conclusion = 'убедительного доказательства роста конверсии нет'
    print(f"{r['variant']}: {conclusion}; изменение выручки на клиента = ${r['revenue_lift_per_user_$']:.2f}.")

print('Рекомендация: оценить стоимость и маржинальность рассылки перед масштабированием.')

Mens E-Mail: рост конверсии статистически значим; изменение выручки на клиента = $0.77.
Womens E-Mail: рост конверсии статистически значим; изменение выручки на клиента = $0.42.
Рекомендация: оценить стоимость и маржинальность рассылки перед масштабированием.
